# Wishes AI - ComfyUI GPU Server (FLUX.2 Klein 4B image edit + Stable Video Diffusion)
Runs on a free Colab **T4**. One ComfyUI instance serves both **image** (FLUX.2 Klein 4B) and **video** (SVD-XT 1.1) through a single Cloudflare tunnel.

**Before running:** add your Hugging Face token to Colab Secrets (key icon in the sidebar) as `HF_TOKEN` with notebook access ON.
Accept the licenses (while logged in to Hugging Face) for `black-forest-labs/FLUX.2-klein-4B` and `stabilityai/stable-video-diffusion-img2vid-xt-1-1`.

Run cells 1 to 4 in order. Do not interrupt a running cell. Keep this tab open.

In [ ]:
# 1. Check GPU and disk
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# 2. Install ComfyUI + cloudflared
%cd /content
!rm -rf ComfyUI
!git clone -q https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb > /dev/null 2>&1
print("ComfyUI + cloudflared installed")

In [ ]:
# 3. Download models (symlinked, so no slow 8 GB copies). Safe to re-run: cached files are not re-downloaded.
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download, list_repo_files, login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
M = "/content/ComfyUI/models"

def fetch(repo, filename, dest_dir):
    files = list_repo_files(repo, token=HF_TOKEN)
    match = [f for f in files if f.split("/")[-1] == filename]
    if not match:
        raise FileNotFoundError(f"{filename} not found in {repo}. Files: {files[:30]}")
    p = hf_hub_download(repo, match[0], token=HF_TOKEN)
    os.makedirs(dest_dir, exist_ok=True)
    dst = os.path.join(dest_dir, filename)
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(p, dst)
    print("OK:", filename, "->", round(os.path.getsize(dst) / 1024**3, 2), "GB")

# Image: FLUX.2 Klein 4B
fetch("black-forest-labs/FLUX.2-klein-4B", "flux-2-klein-4b.safetensors", f"{M}/diffusion_models")
fetch("Comfy-Org/vae-text-encorder-for-flux-klein-4b", "qwen_3_4b.safetensors", f"{M}/text_encoders")
fetch("Comfy-Org/flux2-dev", "flux2-vae.safetensors", f"{M}/vae")

# Video: Stable Video Diffusion XT 1.1
fetch("stabilityai/stable-video-diffusion-img2vid-xt-1-1", "svd_xt_1_1.safetensors", f"{M}/checkpoints")

print("\nAll models ready.")

In [ ]:
# 4. Start ComfyUI + Cloudflare tunnel (prints COMFY_API_URL for backend/.env)
import subprocess, time, re, os, urllib.request
os.system("pkill -f 'main.py' > /dev/null 2>&1; pkill -f cloudflared > /dev/null 2>&1")
time.sleep(2)

subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
                 cwd="/content/ComfyUI",
                 stdout=open("/content/comfy.log", "w"), stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/comfy.log").read()[-3000:])
    raise RuntimeError("ComfyUI did not start - see log above")
print("ComfyUI is up on port 8188")

subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                 stdout=open("/content/cf.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
    if m:
        url = m.group(0)
        break

print("\n" + "=" * 60)
print("COMFY_API_URL=" + str(url))
print("=" * 60)
print("Open that URL in your browser to use ComfyUI directly.")
print("Paste it into backend/.env as COMFY_API_URL, then restart `npm run dev`.")

In [ ]:
# 5. (Optional) Show recent ComfyUI log if something fails, and check VRAM
!tail -n 40 /content/comfy.log
import torch
print("Free VRAM: %.2f GB" % (torch.cuda.mem_get_info()[0] / 1024**3))

## After the tunnel is up
1. Open the `COMFY_API_URL` link, go to **Workflow > Browse Templates**, open **FLUX.2 Klein 4B Distilled: Image Edit**, run it once with any photo.
2. **Workflow > Export (API)** and save as `backend/workflows/flux_klein_api.json`.
3. Put the LoadImage node ID and the positive prompt node ID in `backend/.env` (`COMFY_IMAGE_NODE`, `COMFY_PROMPT_NODE`).
4. Video needs no export: the backend builds the SVD graph itself.

Every new Colab session gives a **new URL**: update `COMFY_API_URL` and restart the backend.